R-CNN Family – Detailed Study & Implementation Guide

The original R-CNN family was proposed by Ross Girshick and fundamentally changed modern object detection.

1️⃣ Objectives of using Selective Search in R-CNN

The objective of Selective Search is:

to generate a small set of high-quality candidate object regions (region proposals) instead of scanning the entire image using sliding windows.

Main objectives:

reduce the number of regions to evaluate

increase recall of true objects

make R-CNN computationally feasible

It produces about ~2000 region proposals per image that are likely to contain objects.

2️⃣ Phases involved in R-CNN

R-CNN works in the following stages.

a) Region Proposal

Selective Search generates candidate bounding boxes.

These are class-independent proposals.

Purpose:
→ locate where objects might exist.

b) Warping and Resizing

Each proposed region has a different size.

So every proposal is:

cropped

resized to a fixed size (e.g. 224 × 224)

Purpose:
→ make it compatible with CNN input.

c) Pre-trained CNN architecture

Each resized proposal is passed through a CNN that was pre-trained on ImageNet.

Purpose:
→ extract deep visual features.

d) Pre-trained SVM model

Instead of softmax, R-CNN uses:

one SVM per class

Purpose:
→ classify each region into object categories.

e) Clean-up

After classification:

many overlapping boxes exist

Post-processing is applied using:

non-maximum suppression (NMS)

f) Implementation of bounding box regression

A regression model refines:

the predicted box coordinates

Purpose:
→ improve localization accuracy.

3️⃣ Possible pre-trained CNNs used in R-CNN

Commonly used CNN backbones:

AlexNet

VGG-16

VGG-19

ResNet-50

ResNet-101

(Deeper backbones give better features but increase cost.)

4️⃣ How is SVM implemented in R-CNN?

In R-CNN:

the CNN is used only as a feature extractor

a separate linear SVM is trained for each class

Workflow:

region → CNN → feature vector → class-specific SVM

Each SVM performs binary classification:
(object vs background for that class).

5️⃣ How does Non-Maximum Suppression (NMS) work?

NMS removes duplicate detections.

Steps:

sort boxes by confidence score

take the highest score box

suppress all boxes with high IoU overlap

repeat

Purpose:
→ keep only the best box per object.

6️⃣ Why Fast R-CNN is better than R-CNN?

R-CNN performs CNN computation for every proposal.

Fast R-CNN:

runs the CNN only once per image

then extracts region features from a shared feature map

Major improvements:

Aspect	R-CNN	Fast R-CNN
CNN runs	~2000 times	1 time
Training	multi-stage	single-stage
Speed	very slow	much faster
7️⃣ Mathematical intuition of ROI Pooling (Fast R-CNN)

ROI pooling converts a variable-sized region into a fixed-size feature map.

Suppose:

ROI width = W

ROI height = H

output size = k × k

Then each bin size is:

bin_w = W / k
bin_h = H / k

For each bin:

output(i,j) = max over all activations inside that bin

So mathematically:

𝑦
𝑖
,
𝑗
=
max
⁡
(
𝑥
,
𝑦
)
∈
𝑏
𝑖
𝑛
(
𝑖
,
𝑗
)
𝐹
(
𝑥
,
𝑦
)
y
i,j
	​

=
(x,y)∈bin(i,j)
max
	​

F(x,y)

This preserves spatial structure while producing fixed-length features.

8️⃣ ROI Projection and ROI Pooling
a) ROI Projection

Original bounding boxes are in image coordinates.

They are mapped to the CNN feature map using the total stride:

x' = x / stride
y' = y / stride

This converts the ROI into feature-map coordinates.

b) ROI Pooling

After projection:

the region is divided into fixed bins

max pooling is applied inside each bin

Output:
→ fixed size tensor for every ROI.

9️⃣ Why object classifier activation changed in Fast R-CNN?

R-CNN:

uses external SVM classifiers

Fast R-CNN:

uses softmax inside the network

Why?

Because Fast R-CNN performs:

joint training

multi-class classification

end-to-end optimization

Softmax allows:

unified loss

backpropagation through the classifier

🔟 Major changes in Faster R-CNN compared to Fast R-CNN

The biggest change is:

👉 introduction of Region Proposal Network (RPN).

Fast R-CNN:

uses Selective Search

Faster R-CNN:

generates proposals using a neural network

Result:

fully deep learning based

much faster

end-to-end trainable

1️⃣1️⃣ Anchor box concept

Anchor boxes are predefined boxes of different:

scales

aspect ratios

placed at every spatial location of the feature map.

The RPN predicts:

which anchors contain objects

how to adjust them

This allows the network to detect:

small

medium

large

wide

tall objects

1️⃣2️⃣ Faster R-CNN – COCO implementation (Notebook ready template)

⚠️ Important:
Training Faster R-CNN from scratch on full COCO requires multiple GPUs and many hours.

For assignment purposes, the correct and accepted approach is:

use a pre-trained Faster R-CNN

fine-tune on a subset of COCO

a) Dataset preparation (COCO)

In [ ]:
from torchvision.datasets import CocoDetection
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = CocoDetection(
    root="coco/train2017",
    annFile="coco/annotations/instances_train2017.json",
    transform=transform
)

val_dataset = CocoDetection(
    root="coco/val2017",
    annFile="coco/annotations/instances_val2017.json",
    transform=transform
)

b) Model architecture

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

model = fasterrcnn_resnet50_fpn(pretrained=True)

num_classes = 91   # COCO classes including background

in_features = model.roi_heads.box_predictor.cls_score.in_features

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features, num_classes
)

c) Training

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params, lr=0.005, momentum=0.9, weight_decay=0.0005
)

Training loop (simplified):

In [ ]:
model.train()

for images, targets in train_loader:

    images = [img.to(device) for img in images]

    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

    loss_dict = model(images, targets)

    losses = sum(loss for loss in loss_dict.values())

    optimizer.zero_grad()
    losses.backward()
    optimizer.step()

Loss used internally

Faster R-CNN already combines:

RPN classification loss

RPN regression loss

ROI classification loss

ROI regression loss

Data augmentation

Use:

In [ ]:
transforms.RandomHorizontalFlip(0.5)
transforms.RandomResizedCrop(...)

d) Validation and mAP

For COCO mAP evaluation:

In [ ]:
d) Validation and mAP

For COCO mAP evaluation:

You must:

store predictions in COCO format

run COCOeval

(This is standard and accepted in assignments.)

e) Inference and visualisation

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model([image.to(device)])

Draw boxes using OpenCV / matplotlib.

f) Optional enhancements

NMS is already applied internally

try different backbones:

ResNet-101

MobileNet

| Model        | Proposal source  | Classifier | End-to-end |
| ------------ | ---------------- | ---------- | ---------- |
| R-CNN        | Selective Search | SVM        | ❌          |
| Fast R-CNN   | Selective Search | Softmax    | ✔️         |
| Faster R-CNN | RPN              | Softmax    | ✔️✔️       |
